#LIBRARIES


#### 1. Environment Setup & Package Installation
Install required dependencies for data processing, image manipulation, and deep learning framework operations.

In [ ]:
pip install pandas numpy opencv-python pillow matplotlib seaborn scikit-learn tqdm


#### 2. Import Libraries & Configure Environment
Import necessary Python packages for data manipulation, computer vision, data visualization, and model development.

In [ ]:
import os

print(os.listdir("/kaggle/input"))


In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
for dirname, dirs, files in os.walk("/kaggle/input"):
    print("\nFolder:", dirname)
    print("Subfolders:", dirs[:10])
    print("Files:", files[:10])


In [ ]:
from pathlib import Path
import pandas as pd
# Define paths based on Kaggle input directory
DATA_ROOT = Path(
    "/kaggle/input/datasets/surajghuwalewala/ham1000-segmentation-and-classification"
)

print("Dataset exists:", DATA_ROOT.exists())
print("Dataset path:", DATA_ROOT)


## Exploratory Data Analysis (EDA)
Load the ground truth labels and inspect the class distribution, missing values, and dataset size.

In [ ]:
# Load dataset metadata
metadata_path = DATA_ROOT / "GroundTruth.csv"

df = pd.read_csv(metadata_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# print("\nFirst 5 rows:")
# display(df.head())


In [ ]:
print(df.info())


In [ ]:
print(df.head(10))


In [ ]:
for root, dirs, files in os.walk(DATA_ROOT):
    print("\nFolder:", root)
    print("Subfolders:", dirs[:10])
    print("Files:", files[:10])

In [ ]:
label_columns = df.columns[1:].tolist()


print("Label columns:", label_columns)

In [ ]:
# Check each class
classes = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]

for col in classes:
    print(f"{col}: {df[col].sum()}")


In [ ]:
# ============================================================
# LABEL PREPROCESSING
# ============================================================

# HAM10000 classes present in GroundTruth.csv
classes = [
    "MEL",
    "NV",
    "BCC",
    "AKIEC",
    "BKL",
    "DF",
    "VASC"
]

# Check that all expected label columns exist
missing_classes = [c for c in classes if c not in df.columns]

if missing_classes:
    raise ValueError(
        f"Missing label columns in GroundTruth.csv: {missing_classes}"
    )

# Make sure label columns are numeric
df[classes] = df[classes].apply(
    pd.to_numeric,
    errors="coerce"
)

# Check for invalid/missing label values
print("Missing values in label columns:")
print(df[classes].isna().sum())

# Fill missing label values with 0
df[classes] = df[classes].fillna(0)

# Check that every image has exactly one class
df["label_sum"] = df[classes].sum(axis=1)

print("\nLabel sum distribution:")
print(df["label_sum"].value_counts())

# Keep only rows having exactly one positive class
df = df[df["label_sum"] == 1].copy()

# Convert one-hot encoded labels into class name
df["label"] = df[classes].idxmax(axis=1)

# Convert class name to numerical ID
from sklearn.preprocessing import LabelEncoder

# Create label encoder
label_encoder = LabelEncoder()

# assign numerical labels
df["label_id"] = label_encoder.fit_transform(df["label"])

#  create mappings
label_map = {
    label: int(label_id)
    for label, label_id in zip(
        label_encoder.classes_,
        label_encoder.transform(label_encoder.classes_)
    )
}

# Reverse mapping – useful later during prediction
id_to_label = {
    v: k for k, v in label_map.items()
}

print("\nClass mapping:")
for class_name, class_id in label_map.items():
    print(f"{class_id} -> {class_name}")

print("\nClass distribution:")
print(df["label"].value_counts())

print("\nFirst 10 records:")
display(
    df[["image", "label", "label_id"]].head(10)
)

In [ ]:
#Check the image files
from pathlib import Path

image_extensions = {".jpg", ".jpeg", ".png"}

image_files = []

for path in DATA_ROOT.rglob("*"):
    if path.is_file() and path.suffix.lower() in image_extensions:
        image_files.append(path)

print("Total image files found:", len(image_files))

print("\nFirst 20 images:")
for path in image_files[:20]:
    print(path)


In [ ]:
mask_files = []

for path in DATA_ROOT.rglob("*"):
    if path.is_file() and path.suffix.lower() in image_extensions:
        if "mask" in str(path).lower() or "segment" in str(path).lower():
            mask_files.append(path)

print("Possible mask files:", len(mask_files))

for path in mask_files[:20]:
    print(path)


In [ ]:
#Check the image names against the CSV
print("CSV image examples:")
print(df["image"].head(10).tolist())


In [ ]:
print("\nActual image examples:")

for path in image_files[:10]:
    print(path.name)


In [ ]:
image_path_dict = {
    path.stem: str(path)
    for path in image_files
}

df["image_path"] = df["image"].map(image_path_dict)

print(df[["image", "image_path"]].head(10))


In [ ]:
print(
    "Images found:",
    df["image_path"].notna().sum()
)

print(
    "Images missing:",
    df["image_path"].isna().sum()
)


In [ ]:
#Remove invalid records
df = df.dropna(
    subset=["image_path"]
).copy()

print("Remaining images:", len(df))


In [ ]:
#Check for corrupted images
from PIL import Image
from tqdm import tqdm

bad_images = []

for path in tqdm(
    df["image_path"],
    desc="Checking images"
):
    try:
        with Image.open(path) as img:
            img.verify()

    except Exception:
        bad_images.append(path)

print("Corrupted images:", len(bad_images))


In [ ]:
#Check image dimensions
from collections import Counter

sizes = []

for path in tqdm(
    df["image_path"],
    desc="Checking dimensions"
):

    with Image.open(path) as img:
        sizes.append(img.size)

print("Most common image sizes:")

for size, count in Counter(sizes).most_common(10):
    print(size, ":", count)


# Image Visualization
Display sample images from the dataset along with their corresponding ground truth annotations.

In [ ]:
#Visualize some images
import matplotlib.pyplot as plt
from PIL import Image

sample_df = df.sample(
    12,
    random_state=42
)

fig, axes = plt.subplots(
    3,
    4,
    figsize=(14, 10)
)

for ax, (_, row) in zip(
    axes.flatten(),
    sample_df.iterrows()
):

    image = Image.open(
        row["image_path"]
    ).convert("RGB")

    ax.imshow(image)

    ax.set_title(
        f"{row['label']}\nID: {row['image']}",
        fontsize=10
    )

    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
#Check class distribution visually
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

sns.countplot(
    data=df,
    x="label",
    order=df["label"].value_counts().index
)

plt.title(
    "HAM10000 Class Distribution"
)

plt.xlabel("Skin Lesion Class")
plt.ylabel("Number of Images")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
#display percentages
class_distribution = df["label"].value_counts()

class_percentage = (
    df["label"].value_counts(normalize=True) * 100
).round(2)

distribution_table = pd.DataFrame({
    "Count": class_distribution,
    "Percentage": class_percentage
})

display(distribution_table)

In [ ]:
#Save our cleaned metadata
from pathlib import Path

PROCESSED_DIR = Path("/kaggle/working/skinova_processed")

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


In [ ]:
df.to_csv(
    PROCESSED_DIR / "cleaned_metadata.csv",
    index=False
)

print(
    "Saved to:",
    PROCESSED_DIR / "cleaned_metadata.csv"
)


In [ ]:
print("Number of CSV rows:", len(df))

print("\nCSV image names:")
print(df["image"].head(10).tolist())

print("\nNumber of image files found:", len(image_files))

print("\nActual image files:")
for path in image_files[:20]:
    print(path)

print("\nNumber of mask files found:", len(mask_files))

print("\nActual mask files:")
for path in mask_files[:20]:
    print(path)


In [ ]:
# ============================================================
# TRAIN / VALIDATION / TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

# First split:
# 80% training
# 20% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label_id"],
    random_state=42
)

# Second split:
# 10% validation
# 10% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label_id"],
    random_state=42
)

# Reset indexes
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training samples  :", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples   :", len(test_df))

In [ ]:
print("\nTraining distribution:")
print(train_df["label"].value_counts())

print("\nValidation distribution:")
print(val_df["label"].value_counts())

print("\nTest distribution:")
print(test_df["label"].value_counts())

# Data Augmentation

In [ ]:
!pip install -q albumentations


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import albumentations as A
from pathlib import Path
from tqdm import tqdm

In [ ]:
# ============================================================
# AUGMENTATION PIPELINE
# ============================================================

augmentation_pipeline = A.Compose([
    
    # Geometric transformations
    A.HorizontalFlip(p=0.5),
    A.Rotate(
        limit=15,
        border_mode=cv2.BORDER_REFLECT_101,
        p=0.5
    ),
    
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.10,
        rotate_limit=0,
        border_mode=cv2.BORDER_REFLECT_101,
        p=0.4
    ),
    
    # Mild color/intensity transformations
    A.RandomBrightnessContrast(
        brightness_limit=0.15,
        contrast_limit=0.15,
        p=0.4
    ),
    
    # Slight image quality variation
    A.GaussianBlur(
        blur_limit=(3, 5),
        p=0.15
    )
])

In [ ]:
# ============================================================
# CREATE AUGMENTED DATASET DIRECTORY
# ============================================================

AUGMENTED_DIR = Path("/kaggle/working/skinova_augmented")

TRAIN_AUG_DIR = AUGMENTED_DIR / "train"

TRAIN_AUG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Augmented dataset directory:")
print(TRAIN_AUG_DIR)

In [ ]:
# ============================================================
# TRAINING CLASS DISTRIBUTION BEFORE AUGMENTATION
# ============================================================

train_distribution = train_df["label"].value_counts().sort_index()

print("Training distribution BEFORE augmentation:")
print(train_distribution)

In [ ]:
# ============================================================
# AUGMENTATION TARGET
# ============================================================

TARGET_IMAGES_PER_CLASS = 2000

print(
    "Target images per minority class:",
    TARGET_IMAGES_PER_CLASS
)

In [ ]:
# ============================================================
# GENERATE AUGMENTED TRAINING IMAGES
# ============================================================

augmented_records = []

# Keep track of the original training images
for _, row in train_df.iterrows():
    
    augmented_records.append({
        "image": row["image"],
        "image_path": row["image_path"],
        "label": row["label"],
        "label_id": row["label_id"],
        "is_augmented": 0
    })


# Process each class separately
for class_name in train_df["label"].unique():
    
    class_df = train_df[
        train_df["label"] == class_name
    ].copy()
    
    current_count = len(class_df)
    
    # Number of additional images required
    images_to_generate = max(
        0,
        TARGET_IMAGES_PER_CLASS - current_count
    )
    
    print(
        f"\nClass: {class_name}"
    )
    print(
        f"Original images: {current_count}"
    )
    print(
        f"Augmented images required: {images_to_generate}"
    )
    
    if images_to_generate == 0:
        continue
    
    # Generate augmented images
    for i in tqdm(
        range(images_to_generate),
        desc=f"Augmenting {class_name}"
    ):
        
        # Randomly select an original image from this class
        row = class_df.sample(
            n=1,
            random_state=42 + i
        ).iloc[0]
        
        image_path = row["image_path"]
        
        # Read image
        image = cv2.imread(image_path)
        
        if image is None:
            print(
                f"Warning: Could not read {image_path}"
            )
            continue
        
        # Convert BGR -> RGB
        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )
        
        # Apply augmentation
        augmented = augmentation_pipeline(
            image=image
        )
        
        augmented_image = augmented["image"]
        
        # Convert RGB -> BGR for saving
        augmented_image = cv2.cvtColor(
            augmented_image,
            cv2.COLOR_RGB2BGR
        )
        
        # Create unique filename
        original_name = Path(
            row["image_path"]
        ).stem
        
        new_filename = (
            f"{original_name}_aug_{i:05d}.jpg"
        )
        
        output_path = (
            TRAIN_AUG_DIR /
            new_filename
        )
        
        # Save augmented image
        cv2.imwrite(
            str(output_path),
            augmented_image,
            [cv2.IMWRITE_JPEG_QUALITY, 95]
        )
        
        # Store metadata
        augmented_records.append({
            "image": new_filename,
            "image_path": str(output_path),
            "label": row["label"],
            "label_id": row["label_id"],
            "is_augmented": 1
        })

print("\nData augmentation completed.")

In [ ]:
# ============================================================
# CREATE AUGMENTED TRAINING METADATA
# ============================================================

augmented_train_df = pd.DataFrame(
    augmented_records
)

print(
    "Total training images after augmentation:",
    len(augmented_train_df)
)

print("\nClass distribution AFTER augmentation:")
print(
    augmented_train_df["label"].value_counts()
)

In [ ]:
# ============================================================
# VISUALIZE AUGMENTED IMAGES
# ============================================================

import matplotlib.pyplot as plt

# Select augmented images only
sample_aug = augmented_train_df[
    augmented_train_df["is_augmented"] == 1
].sample(
    8,
    random_state=42
)

fig, axes = plt.subplots(
    2,
    4,
    figsize=(14, 7)
)

for ax, (_, row) in zip(
    axes.flatten(),
    sample_aug.iterrows()
):
    
    image = cv2.imread(
        row["image_path"]
    )
    
    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )
    
    ax.imshow(image)
    
    ax.set_title(
        row["label"]
    )
    
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CLASS DISTRIBUTION AFTER AUGMENTATION
# ============================================================

plt.figure(figsize=(10, 6))

augmented_train_df["label"].value_counts().plot(
    kind="bar"
)

plt.title(
    "Training Class Distribution After Data Augmentation"
)

plt.xlabel("Skin Disease Class")
plt.ylabel("Number of Images")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SAVE AUGMENTED METADATA
# ============================================================

augmented_metadata_path = (
    AUGMENTED_DIR /
    "augmented_train_metadata.csv"
)

augmented_train_df.to_csv(
    augmented_metadata_path,
    index=False
)

print(
    "Augmented metadata saved to:"
)

print(
    augmented_metadata_path
)

In [ ]:
# ============================================================
# SAVE VALIDATION AND TEST METADATA
# ============================================================

val_df.to_csv(
    AUGMENTED_DIR / "validation_metadata.csv",
    index=False
)

test_df.to_csv(
    AUGMENTED_DIR / "test_metadata.csv",
    index=False
)

print("Validation metadata saved.")
print("Test metadata saved.")